# The Assignment — Diff-First Ingestion Prompt Validation

Pre-implementation gate for Sprint 3 Feature 1 (KnowledgeUpdateTab).

**3 test cases:**
1. Typed note — policy change: `"Goldman changed their EP policy"`
   - Expect: `profile_updates` for `investment_banking.ep_sponsorship` + possibly `new_chunks`
2. Synthetic document — GIC internship info (simulates 2-page PDF)
   - Expect: `new_chunks` with `career_type: public_sector`, minimal `already_covered`
3. Redundant note: `"Investment banking involves financial analysis"`
   - Expect: `already_covered` non-empty, `new_chunks` empty, `profile_updates` empty
   - **Critical test** — if this fails (Claude hallucinates an update), redesign the prompt

**Pass criteria:** All 3 cases produce expected diff shape. If case 3 produces a non-empty `profile_updates`, redesign before building.

---

**Requires:** API running at `localhost:8000`, `ANTHROPIC_API_KEY` in `.env` or env

In [1]:
import json
import os
import sys
from pathlib import Path

import anthropic
import requests
import yaml

# ── Paths ──────────────────────────────────────────────────────────────────
REPO_ROOT = Path("__file__").resolve().parent.parent
PROFILES_DIR = REPO_ROOT / "knowledge" / "career_profiles"
API_BASE = "http://localhost:8000"

# ── API key ────────────────────────────────────────────────────────────────
api_key = os.environ.get("ANTHROPIC_API_KEY")
if not api_key:
    env_path = REPO_ROOT / ".env"
    if env_path.exists():
        for line in env_path.read_text().splitlines():
            if line.startswith("ANTHROPIC_API_KEY="):
                api_key = line.split("=", 1)[1].strip().strip('"').strip("'")
                break

assert api_key, "ANTHROPIC_API_KEY not found — add to .env or export it"
client = anthropic.Anthropic(api_key=api_key)
print("API key loaded. Checking backend...")

health = requests.get(f"{API_BASE}/health", timeout=5)
assert health.ok, f"API not reachable at {API_BASE}"
print(f"Backend OK: {health.json()}")

API key loaded. Checking backend...
Backend OK: {'status': 'ok'}


In [2]:
# ── Load career profiles → build profile_summary ───────────────────────────

def first_sentence(text: str, max_chars: int = 120) -> str:
    """Extract first sentence up to max_chars (mirrors prompt spec)."""
    if not text:
        return ""
    text = str(text).strip()
    dot = text.find(".")
    if dot != -1 and dot < max_chars:
        return text[: dot + 1]
    return text[:max_chars]


def build_profile_summary(profiles_dir: Path) -> str:
    """Build the CURRENT CAREER PROFILE FIELDS block for the diff prompt.
    
    Format: <slug>: ep_sponsorship=<first sentence> | compass=<val> | ...
    """
    lines = []
    for yaml_path in sorted(profiles_dir.glob("*.yaml")):
        p = yaml.safe_load(yaml_path.read_text())
        slug = yaml_path.stem
        ep = first_sentence(p.get("ep_sponsorship", ""))
        compass = first_sentence(p.get("compass_score_typical", ""))
        timeline = first_sentence(p.get("recruiting_timeline", ""))
        notes = first_sentence(p.get("notes", ""))
        lines.append(
            f"{slug}: ep_sponsorship={ep} | compass={compass} | "
            f"recruiting_timeline={timeline} | notes={notes}"
        )
    return "\n".join(lines)


PROFILE_SUMMARY = build_profile_summary(PROFILES_DIR)
print("Profile summary built:")
print("─" * 60)
for line in PROFILE_SUMMARY.splitlines():
    print(f"  {line[:120]}..." if len(line) > 120 else f"  {line}")

Profile summary built:
────────────────────────────────────────────────────────────
  consulting: ep_sponsorship=High at MBB (McKinsey, BCG, Bain) and Tier 2 global firms (Oliver Wyman, Strategy&, AT Kearne...
  general_singapore: ep_sponsorship=Varies significantly by employer size and industry. | compass=Varies — see specific ca...
  investment_banking: ep_sponsorship=High at bulge-bracket banks (Goldman Sachs, Morgan Stanley, JPMorgan, Citi, Deutsche ...
  public_sector: ep_sponsorship=High for statutory boards and GLCs (GIC, Temasek, MAS, EDB, JTC, HDB, CPF Board, CAAS) — t...
  tech_product: ep_sponsorship=High at large tech MNCs with established Singapore entities (Google, Meta, Amazon, Microsof...


In [3]:
# ── Helper: retrieve top-10 chunks from running KB ─────────────────────────

def retrieve_chunks(query: str, limit: int = 10) -> str:
    """Call /api/kb/test-query and format results for the diff prompt."""
    resp = requests.post(
        f"{API_BASE}/api/kb/test-query",
        json={"query": query, "limit": limit},
        timeout=15,
    )
    resp.raise_for_status()
    chunks = resp.json()
    if not isinstance(chunks, list):
        return "(KB returned no results)"
    lines = []
    for i, c in enumerate(chunks, 1):
        src = c.get("source_filename", "unknown")
        score = c.get("score", 0)
        excerpt = c.get("excerpt", "").replace("\n", " ").strip()
        lines.append(f"[{i}] (score={score:.3f}) source={src}\n{excerpt}")
    return "\n\n".join(lines)


# Quick sanity check
sample = retrieve_chunks("Goldman EP sponsorship")
print(f"Retrieved {len(sample.splitlines())} lines for sanity query")
print(sample[:400], "...")

Retrieved 14 lines for sanity query
[1] (score=0.340) source=goldman-singapore-guide.txt
Goldman Sachs Singapore — SMU Campus Recruiting (2025-2026)  DIVISIONS HIRING FROM SMU Investment Banking Division (IBD): 4-6 analysts/year from SMU Securities (Equities/FICC): 3-5 analysts/year Engineering (Technology): 6-10 engineers/year Asset Management: 2-3 analysts/year  IBD APPLICATION PROCES

[2] (score=0.251) source=smu-alumni-paths.txt ...


In [4]:

# ── The diff prompt (verbatim from design doc Architecture section) ─────────

SYSTEM_PROMPT = """You are a knowledge base curator for {school_name}'s career advisory AI.
        You will be given new input text and excerpts from the existing knowledge base.
        Your task: identify what the input adds, changes, or contradicts relative to
        the existing KB. Return ONLY a JSON object matching the schema below.
        Do not add explanatory text outside the JSON.

        Output schema:
        {{
          "interpretation_bullets": ["<2-5 short bullets — plain English>"],
          "profile_updates": {{
            "<career_type_slug>": {{
              "<field_name>": {{ "old": "<current value or null>", "new": "<proposed>" }}
            }}
          }},
          "new_chunks": [
            {{ "text": "<chunk>", "source_type": "note", "source_label": "counsellor_note",
              "career_type": "<slug or null>", "chunk_id": "" }}
          ],
          "already_covered": [
            {{ "excerpt": "<excerpt>", "source_doc": "<filename>" }}
          ]
        }}

        Rules:
        - Only propose profile_updates for fields that the input clearly changes or
          corrects. Use the exact field names from CURRENT CAREER PROFILE FIELDS.
          Do not guess field names not listed there.
        - new_chunks: self-contained facts not present in existing excerpts.
          Maximum 3 chunks. Prefer fewer, denser chunks. Leave chunk_id as "".
        - already_covered: existing KB excerpts substantially overlapping with the
          input. Include up to 5.
        - career_type: use the slug from CURRENT CAREER PROFILE FIELDS, or null."""

USER_PROMPT_TEMPLATE = """INPUT TEXT:
{counsellor_input}

EXISTING KB EXCERPTS (top 10 by semantic similarity):
{retrieved_chunks_formatted}

CURRENT CAREER PROFILE FIELDS (key fields only):
{profile_summary}
Format per profile: "<slug>: ep_sponsorship=<first sentence> | compass=<val> | ..."
Include: ep_sponsorship (first sentence only), compass_score_typical,
recruiting_timeline (first sentence), notes (first sentence)."""


SCHOOL_NAME = "SMU (Singapore Management University)"


def run_diff_prompt(counsellor_input: str, query_for_retrieval: str | None = None) -> dict:
    """Run the full diff prompt against Claude. Returns parsed JSON dict."""
    query = query_for_retrieval or counsellor_input
    retrieved = retrieve_chunks(query)
    
    system = SYSTEM_PROMPT.format(school_name=SCHOOL_NAME)
    user = USER_PROMPT_TEMPLATE.format(
        counsellor_input=counsellor_input,
        retrieved_chunks_formatted=retrieved,
        profile_summary=PROFILE_SUMMARY,
    )
    
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=2048,
        system=system,
        messages=[{"role": "user", "content": user}],
    )
    raw = response.content[0].text.strip()
    
    # Strip markdown code fences (```json\n...\n``` or ```\n...\n```)
    if raw.startswith("```"):
        # Drop the opening fence line (e.g. "```json")
        raw = raw.split("\n", 1)[1]
        # Drop everything from the last closing fence onward
        if "```" in raw:
            raw = raw.rsplit("```", 1)[0]
    raw = raw.strip()
    
    return json.loads(raw)


print("Prompt helpers ready.")


Prompt helpers ready.


In [5]:
# ── Grader ─────────────────────────────────────────────────────────────────

def grade(label: str, result: dict, expect: dict) -> None:
    """Print a pass/fail breakdown for a test case.
    
    expect keys:
        profile_updates_slugs: list of slugs that MUST have at least one update
        profile_updates_empty: bool — profile_updates must be {}
        new_chunks_min: int — minimum new_chunks count
        new_chunks_empty: bool — new_chunks must be []
        new_chunks_career_type: str — all new_chunks must have this career_type
        already_covered_nonempty: bool — already_covered must have at least 1 entry
    """
    passes = []
    fails = []

    pu = result.get("profile_updates", {})
    nc = result.get("new_chunks", [])
    ac = result.get("already_covered", [])

    if "profile_updates_slugs" in expect:
        for slug in expect["profile_updates_slugs"]:
            if slug in pu and pu[slug]:
                passes.append(f"profile_updates has entry for '{slug}'")
            else:
                fails.append(f"profile_updates MISSING entry for '{slug}' (got keys: {list(pu.keys())})")

    if expect.get("profile_updates_empty"):
        if not pu:
            passes.append("profile_updates is empty (correct — no hallucinated updates)")
        else:
            fails.append(f"profile_updates NOT empty — hallucinated update(s): {list(pu.keys())}")

    if "new_chunks_min" in expect:
        if len(nc) >= expect["new_chunks_min"]:
            passes.append(f"new_chunks count={len(nc)} >= {expect['new_chunks_min']}")
        else:
            fails.append(f"new_chunks count={len(nc)} < {expect['new_chunks_min']}")

    if expect.get("new_chunks_empty"):
        if not nc:
            passes.append("new_chunks is empty (correct — all content already covered)")
        else:
            fails.append(f"new_chunks NOT empty — should be empty. Got: {[c.get('text','')[:60] for c in nc]}")

    if "new_chunks_career_type" in expect:
        wrong = [c for c in nc if c.get("career_type") != expect["new_chunks_career_type"]]
        if not wrong:
            passes.append(f"all new_chunks have career_type='{expect['new_chunks_career_type']}'")
        else:
            fails.append(
                f"{len(wrong)} chunk(s) have wrong career_type: "
                f"{[c.get('career_type') for c in wrong]}"
            )

    if expect.get("already_covered_nonempty"):
        if ac:
            passes.append(f"already_covered has {len(ac)} entry/entries")
        else:
            fails.append("already_covered is empty — expected overlap detection")

    status = "PASS" if not fails else "FAIL"
    bar = "=" * 60
    print(f"\n{bar}")
    print(f"TEST: {label}  [{status}]")
    print(bar)
    for p in passes:
        print(f"  ✓  {p}")
    for f in fails:
        print(f"  ✗  {f}")
    return status

print("Grader ready.")

Grader ready.


## Test 1 — Policy change note

Input: a specific EP policy change from Goldman.

**Note on test design:** A vague input like `"Goldman changed their EP policy"` intentionally fails — Claude correctly refuses to hallucinate a direction. The real gate is whether the prompt generates a `profile_updates` entry when the counsellor gives *specific* new information. So test 1 uses a realistic specific note.

Expected:
- `profile_updates[investment_banking]` has an `ep_sponsorship` update
- `new_chunks` may be empty (change is a profile field update, not a new fact)
- `already_covered` has entries from existing Goldman docs

In [6]:

INPUT_1 = (
    "Goldman Sachs Singapore has updated their EP sponsorship policy for the 2026 hiring cycle. "
    "They are now only sponsoring EP for international candidates who score 50+ on COMPASS. "
    "Previously they would sponsor anyone above 40 (the minimum). "
    "Students below 50 should now target DBS or OCBC instead."
)

result_1 = run_diff_prompt(INPUT_1)

print("Raw output:")
print(json.dumps(result_1, indent=2))


Raw output:
{
  "interpretation_bullets": [
    "Goldman Sachs Singapore has raised its internal EP sponsorship threshold from 40 (the MOM minimum) to 50+ on COMPASS for the 2026 hiring cycle.",
    "This directly updates the investment_banking profile's compass and ep_sponsorship fields, which currently cite 45\u201355 as typical but do not specify a GS-specific cutoff.",
    "The note recommends redirecting international students below 50 COMPASS toward DBS or OCBC, which aligns with existing KB guidance on local bank recruiting.",
    "No other banks are mentioned as changing policy \u2014 this is a Goldman Sachs-specific update."
  ],
  "profile_updates": {
    "investment_banking": {
      "compass": {
        "old": "45\u201355 for analyst roles at BBs (S$75K\u201390K starting salary anchors the salary criterion at roughly 20\u201325 pts; degree from ...",
        "new": "50\u201355 for analyst roles at BBs. Note: Goldman Sachs Singapore has set an internal EP sponsorship floor o

In [7]:
grade(
    "Test 1 — Policy change note",
    result_1,
    {
        "profile_updates_slugs": ["investment_banking"],
    },
)


TEST: Test 1 — Policy change note  [PASS]
  ✓  profile_updates has entry for 'investment_banking'


'PASS'

## Test 2 — Synthetic GIC internship document

Simulates uploading a 2-page PDF about GIC internship applications.
Text is new content not present in the KB.

Expected:
- `new_chunks` has 1+ entries with `career_type: public_sector`
- `profile_updates` may be empty or have `public_sector` updates — acceptable
- `already_covered` may have partial overlaps with existing public sector content

In [8]:
# Synthetic 2-page PDF content — new facts not in existing KB
INPUT_2 = """\
GIC Internship Programme 2025-2026 — Information for SMU Students

GIC is hiring 12-week summer interns (June–August 2026) in the following divisions:
  - Fixed Income & Multi-Asset (FIMA): 8 positions
  - Equities: 5 positions
  - Infrastructure & Private Equity: 3 positions
  - Technology: 10 positions

Application opens: 1 October 2025
Application closes: 15 November 2025 (strict deadline — late applications not reviewed)

Key changes for 2025-2026 cycle:
- GIC has removed the video interview round and now conducts 2 technical interviews directly
  after the online application screening.
- All offers include EP sponsorship confirmed in writing at the offer stage.
- Minimum GPA requirement raised to 3.7 (was 3.5) for investment division roles.

Compensation: S$3,800/month for all divisions. Housing allowance of S$500/month for
non-local students who need to relocate within Singapore.

SMU career centre note: GIC has reserved 4 dedicated slots for SMU students in the FIMA
division. Students who apply through the SMU portal (not GIC's public portal) will receive
an SMU referral tag — use this if you want to be considered for the reserved slots.

Return offer rate: approximately 60% of interns receive full-time offers. Full-time starts
in July 2027 for the 2026 cohort.

For questions, contact the SMU career centre's public sector track advisor
(details via the internal portal).
"""

result_2 = run_diff_prompt(
    INPUT_2,
    query_for_retrieval="GIC internship public sector Singapore government investment",
)

print("Raw output:")
print(json.dumps(result_2, indent=2))

Raw output:
{
  "interpretation_bullets": [
    "GIC has updated its 2025-2026 internship cycle: video interview round removed, replaced by 2 direct technical interviews post-screening.",
    "Minimum GPA for investment division roles raised from 3.5 to 3.7; compensation set at S$3,800/month plus S$500 housing allowance for non-locals.",
    "SMU students have 4 reserved FIMA slots accessible only via the SMU portal (not GIC's public portal), which adds an SMU referral tag.",
    "Application window is 1 Oct\u201315 Nov 2025 (strict); return offer rate ~60%; full-time starts July 2027 for 2026 cohort.",
    "All GIC offers now include EP sponsorship confirmed in writing at offer stage \u2014 a notable process improvement worth surfacing to students."
  ],
  "profile_updates": {
    "public_sector": {
      "recruiting_timeline": {
        "old": "Management Associate (MA) programmes: applications typically open August\u2013October for a July/August start the following y",
        "new"

In [9]:
grade(
    "Test 2 — GIC internship document",
    result_2,
    {
        "new_chunks_min": 1,
        "new_chunks_career_type": "public_sector",
    },
)


TEST: Test 2 — GIC internship document  [PASS]
  ✓  new_chunks count=1 >= 1
  ✓  all new_chunks have career_type='public_sector'


'PASS'

## Test 3 — Redundant note (critical test)

Input: `"Investment banking involves financial analysis"`

This is a vague, obvious statement already well-covered in the KB.

Expected:
- `already_covered` non-empty
- `new_chunks` **empty** — no new content to add
- `profile_updates` **empty** — no field update (this would be hallucination)

**If `profile_updates` is non-empty: redesign the prompt before building Feature 1.**

In [10]:
INPUT_3 = "Investment banking involves financial analysis"

result_3 = run_diff_prompt(INPUT_3)

print("Raw output:")
print(json.dumps(result_3, indent=2))

Raw output:
{
  "interpretation_bullets": [
    "Input states a basic, generic fact: 'Investment banking involves financial analysis'.",
    "This is common knowledge already implied by the extensive investment banking content in the KB.",
    "No new actionable information, corrections, or field updates are introduced.",
    "No contradictions with existing KB content detected."
  ],
  "profile_updates": {},
  "new_chunks": [],
  "already_covered": [
    {
      "excerpt": "Q: I want to go into investment banking. What should I do in Year 1? A: Join SMU Investment Club immediately. Take FNCE321 (Corporate Finance) early. Start reading Financial Times and Bloomberg daily. Get comfortable with Excel.",
      "source_doc": "career-office-faq.txt"
    },
    {
      "excerpt": "SMU Lee Kong Chian School of Business (Finance) \u2192 Goldman Sachs Singapore (IBD Analyst) Typical path: Strong GPA (3.8+), CFA Level 1 before graduation, SMU Investment Club leadership.",
      "source_doc": "sm

In [11]:
grade(
    "Test 3 — Redundant note [CRITICAL]",
    result_3,
    {
        "already_covered_nonempty": True,
        "new_chunks_empty": True,
        "profile_updates_empty": True,
    },
)


TEST: Test 3 — Redundant note [CRITICAL]  [PASS]
  ✓  profile_updates is empty (correct — no hallucinated updates)
  ✓  new_chunks is empty (correct — all content already covered)
  ✓  already_covered has 3 entry/entries


'PASS'

## Verdict

Run the cell below after all 3 tests to see the overall result.

In [12]:
# Re-run grades and collect statuses
statuses = [
    grade("Test 1 — Policy change note", result_1,
          {"profile_updates_slugs": ["investment_banking"]}),
    grade("Test 2 — GIC internship document", result_2,
          {"new_chunks_min": 1, "new_chunks_career_type": "public_sector"}),
    grade("Test 3 — Redundant note [CRITICAL]", result_3,
          {"already_covered_nonempty": True, "new_chunks_empty": True,
           "profile_updates_empty": True}),
]

print("\n" + "=" * 60)
all_pass = all(s == "PASS" for s in statuses)
if all_pass:
    print("VERDICT: ALL PASS — build Feature 1")
    print("The diff prompt reliably identifies new content, field updates, and redundant input.")
else:
    failed = [i+1 for i, s in enumerate(statuses) if s != "PASS"]
    print(f"VERDICT: FAIL — test(s) {failed} did not pass")
    if 3 in failed:
        print("CRITICAL: Test 3 failed — prompt hallucinates updates on redundant input.")
        print("Redesign the prompt before implementing Feature 1.")
    else:
        print("Review failures above. Consider prompt adjustments before building.")
print("=" * 60)


TEST: Test 1 — Policy change note  [PASS]
  ✓  profile_updates has entry for 'investment_banking'

TEST: Test 2 — GIC internship document  [PASS]
  ✓  new_chunks count=1 >= 1
  ✓  all new_chunks have career_type='public_sector'

TEST: Test 3 — Redundant note [CRITICAL]  [PASS]
  ✓  profile_updates is empty (correct — no hallucinated updates)
  ✓  new_chunks is empty (correct — all content already covered)
  ✓  already_covered has 3 entry/entries

VERDICT: ALL PASS — build Feature 1
The diff prompt reliably identifies new content, field updates, and redundant input.
